In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### I compared the station_name but didn't compare the native_id. While the raws insertaion based on native_id to recongnize station. 

Because I didn't compare native_id before inserting, and the native_id exist many mistach, thus caused some confusion.

It can be mainly devided into two parts, the first batch is different id, same station_name; the second batch is different id, different 


Thus, only the data with same native_id has been inserted and merged; data with different native_id are recognized as different stations.

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [15]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
# from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [16]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [17]:
# --- Main workflow ---
# --- read data ---
meta_path = '/workspaces/crmprtd/fern/FERNNorth2025_insert/tables/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
os.makedirs(output_folder, exist_ok=True)


In [18]:

native_ids = df_station['native_id'].tolist()
station_names = df_station['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    existing_stations = pd.read_sql(query, conn, params={"station_names": station_names})

print("Stations found in DB:")
existing_stations

Stations found in DB:


,station_name,native_id
0,Atlin School,AtlSch
1,BarrenWx,Barren
2,BlackhawkWx,Blackhawk
3,BoulderWx,Boulder Creek
4,BowronPit,BowronPit
5,BulkleyWx,PGTIS1
6,Canoe Mountain Stn,Canoe Mountain Stn
7,Caribou Pass,CaribouPass
8,ChapmanWx,Chapman
9,ChiefLakeWx,ChiefWx


In [ ]:
import os
import pandas as pd
import re
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'

# --- Load the CSV ---
compare_file_path = os.path.join(output_folder, "Fern_raw_db_station_matched.csv")
compare_summary = pd.read_csv(compare_file_path)

# --- Step 1: Define base variable (remove trailing digits) ---
def base_variable(var):
    return re.sub(r'\s*\d+$', '', var).strip()

compare_summary['base_variable'] = compare_summary['variable'].apply(base_variable)

# --- Step 2: Identify duplicates per station + base_variable + unit_raw ---
group_keys = ['station_name', 'base_variable', 'unit_raw']

# Rows that are duplicates (appear more than once in the group)
dup_items = (
    compare_summary
    .groupby(group_keys)
    .filter(lambda x: len(x) > 1)
    .reset_index(drop=True)
)

# --- Step 3: Identify unique rows (not in duplicates) ---
# Safer approach using merge instead of relying on index
dup_keys = dup_items[group_keys].drop_duplicates()
unique_items = (
    compare_summary
    .merge(dup_keys, on=group_keys, how='left', indicator=True)
    .query('_merge == "left_only"')  # only rows not in dup_items
    .drop(columns='_merge')
)

# --- Step 4: Sort unique items for readability ---
unique_items = unique_items.sort_values(by=['station_name', 'variable']).reset_index(drop=True)


,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,base_variable
0,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,2011-07-28 09:00:00,2024-08-10 19:00:00,0.0,0.0,batch1,DewPt
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,2010-08-20 19:00:00,2024-08-10 19:00:00,0.0,0.0,batch1,Gust Speed
2,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,0.0,0.0,batch1,RH
3,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,0.0,0.0,batch1,Rain
4,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaN,NaN,NaN,NaN,batch4,Sfc Temp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2024-10-24 09:00:00,0.0,371.0,batch1,Solar Radiation
415,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2024-10-24 09:00:00,0.0,371.0,batch1,Temp
416,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2024-10-24 09:00:00,0.0,371.0,batch1,Wetness
417,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2024-10-24 09:00:00,0.0,371.0,batch1,Wind Direction


### Loop for all stations and variables

In [36]:
import os
import pandas as pd

def compare_csv_vs_db(row, data_path, engine):
    """
    Compare one CSV variable with the database.
    Returns a dataframe with Date, csv_value, db_value, diff, match.
    """
    csv_file_name = row['station_file_name'] + '.csv'
    csv_path = os.path.join(data_path, csv_file_name)

    # Load CSV
    df_data = safe_read_csv(csv_path)

    # Time column
    time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    # Variable column
    variable_name = row['variable'] + ', ' + row['unit_origin']
    df_subset = df_data[[time_col, variable_name]].copy()
    df_subset.rename(columns={time_col: 'Date', variable_name: 'csv_value'}, inplace=True)

    # Pull DB data
    db_var_name = row['db_var_match']
    db_native_id = row['native_id_raw']
    query = """
    SELECT o.obs_time, o.datum
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.native_id = %s
      AND v.net_var_name = %s
      AND o.obs_time BETWEEN %s AND %s
    ORDER BY o.obs_time
    """
    params = (
        db_native_id,
        db_var_name,
        df_subset['Date'].min(),
        df_subset['Date'].max()
    )
    df_db = pd.read_sql(query, engine, params=params)
    df_db.rename(columns={'obs_time': 'Date', 'datum': 'db_value'}, inplace=True)

    # Merge CSV & DB
    df_compare = pd.merge(df_subset, df_db, on='Date', how='outer')

    # Keep rows where at least one value exists
    df_compare = df_compare[~(df_compare['csv_value'].isna() & df_compare['db_value'].isna())].copy()

    # Differences & match
    # --- Ensure numeric types for comparison ---
    df_compare['csv_value'] = pd.to_numeric(df_compare['csv_value'], errors='coerce')
    df_compare['db_value']  = pd.to_numeric(df_compare['db_value'], errors='coerce')

    # Differences & match
    df_compare['diff'] = df_compare['csv_value'] - df_compare['db_value']

    # Consider them matching if both are NaN or difference is very small
    df_compare['match'] = df_compare['diff'].fillna(0).abs() < 0.01  # tolerance for floats

    return df_compare


# --- Loop over all rows in unique_items ---
summary_list = []

for i, row in unique_items.iloc[0:].iterrows():
    print(f"\n🔍 Comparing {row['station_name']} — {row['variable']} ({i+1}/{len(unique_items)})")
    df_comp = compare_csv_vs_db(row, data_path, engine)

    num_total = len(df_comp)
    num_mismatch = (~df_comp['match']).sum()
    num_match = df_comp['match'].sum()

    # Get latest timestamp of mismatch, if any
    if num_mismatch > 0:
        latest_mismatch_time = df_comp.loc[~df_comp['match'], 'Date'].max()
        print(f"  ⚠️ {num_mismatch}/{num_total} mismatches, latest mismatch at {latest_mismatch_time}")
    else:
        latest_mismatch_time = pd.NaT
        print(f"  ✅ All {num_total} values match")

    summary_list.append({
        'station': row['station_name'],
        'variable': row['variable'],
        'total_points': num_total,
        'matches': num_match,
        'mismatches': num_mismatch,
        'latest_mismatch': latest_mismatch_time,
        'latest_time_raw': row['latest_time_raw']

    })

# --- Create summary dataframe ---
df_summary = pd.DataFrame(summary_list)

df_summary['mismatch_perc'] = df_summary['mismatches']/ df_summary['total_points']

df_summary
# # Optionally save to CSV
# summary_csv_path = os.path.join(output_folder, "6.Value_check_Fern_raw_db_comparison_summary.csv")
# df_summary.to_csv(summary_csv_path, index=False)

# print("\n=== Comparison Summary ===")
# print(df_summary)
# print(f"\n✅ Summary saved to: {summary_csv_path}")


🔍 Comparing Atlin School — DewPt (1/419)
  ✅ All 111040 values match

🔍 Comparing Atlin School — Gust Speed (2/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — RH (3/419)
  ✅ All 111040 values match

🔍 Comparing Atlin School — Rain (4/419)
  ✅ All 119232 values match

🔍 Comparing Atlin School — Sfc Temp (5/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Soil Temp (6/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Soil Temp 50 cm (7/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Soil Temp 75 cm (8/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Solar Radiation (9/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Wind Direction (10/419)
  ✅ All 119238 values match

🔍 Comparing Atlin School — Wind Speed (11/419)
  ✅ All 119238 values match

🔍 Comparing BarrenWx — DewPt (12/419)
  ✅ All 36912 values match

🔍 Comparing BarrenWx — DewPt 30 cm (13/419)
  ✅ All 30527 values match

🔍 Comparing BarrenWx — Gust Speed (14

,station,variable,total_points,matches,mismatches,latest_mismatch,latest_time_raw,mismatch_perc
0,Atlin School,DewPt,111040,111040,0,NaT,2024-08-10 19:00:00,0.0
1,Atlin School,Gust Speed,119238,119238,0,NaT,2024-08-10 19:00:00,0.0
2,Atlin School,RH,111040,111040,0,NaT,2024-08-10 19:00:00,0.0
3,Atlin School,Rain,119232,119232,0,NaT,2024-08-10 19:00:00,0.0
4,Atlin School,Sfc Temp,119238,119238,0,NaT,2024-08-10 19:00:00,0.0
...,...,...,...,...,...,...,...,...
414,Willow-BowronWx,Solar Radiation,144146,144146,0,NaT,2025-10-30 10:00:00,0.0
415,Willow-BowronWx,Temp,144115,144115,0,NaT,2025-10-30 10:00:00,0.0
416,Willow-BowronWx,Wetness,137632,137632,0,NaT,2025-10-30 10:00:00,0.0
417,Willow-BowronWx,Wind Direction,113204,113204,0,NaT,2025-10-30 10:00:00,0.0


### Single case

In [ ]:
import os
import pandas as pd

# --- Step 0: Setup paths and variables ---
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'
row = unique_items.iloc[348]
csv_file_name = row['station_file_name'] + '.csv'
csv_path = os.path.join(data_path, csv_file_name)

db_var_name = row['db_var_match']
db_station_name = row['station_name']
db_native_id = row['native_id_raw']
print(db_station_name)
print(db_var_name)

# --- Step 1: Load CSV data ---
df_data = safe_read_csv(csv_path)

# Determine the time column
time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

# Construct the variable column name
variable_name = row['variable'] + ', ' + row['unit_origin']

# Keep only time and variable columns
df_subset = df_data[[time_col, variable_name]].copy()
df_subset.rename(columns={variable_name: 'csv_value'}, inplace=True)

# --- Step 2: Pull database data for this station/variable ---
query = """
SELECT o.obs_time, o.datum
FROM obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
JOIN meta_vars v ON o.vars_id = v.vars_id
WHERE s.native_id = %s
  AND v.net_var_name = %s
  AND o.obs_time BETWEEN %s AND %s
ORDER BY o.obs_time
"""

params = (
    db_native_id,
    db_var_name,
    df_subset[time_col].min(),
    df_subset[time_col].max()
)

df_db = pd.read_sql(query, engine, params=params)
df_db.rename(columns={'obs_time': time_col, 'datum': 'db_value'}, inplace=True)

# --- Step 3: Merge CSV and DB data on timestamp ---
df_compare = pd.merge(
    df_subset,
    df_db,
    on=time_col,
    how='outer'  # include missing values from either side
)

# --- Step 4: Filter out rows where both CSV and DB values are NaN ---
df_compare = df_compare[~(df_compare['csv_value'].isna() & df_compare['db_value'].isna())].copy()

# --- Step 5: Compute differences and matches ---
# --- Ensure numeric types for comparison ---
df_compare['csv_value'] = pd.to_numeric(df_compare['csv_value'], errors='coerce')
df_compare['db_value']  = pd.to_numeric(df_compare['db_value'], errors='coerce')

# Differences & match
df_compare['diff'] = df_compare['csv_value'] - df_compare['db_value']

# Consider them matching if both are NaN or difference is very small
df_compare['match'] = df_compare['diff'].fillna(0).abs() < 0.01  # tolerance for floats
# Optional: adjust tolerance for floating point differences
# df_compare['match'] = df_compare['diff'].abs() < 0.01

# --- Step 6: Summary of mismatches ---
num_mismatches = (~df_compare['match']).sum()
print(f"Total rows compared: {len(df_compare)}")
print(f"Number of mismatches: {num_mismatches}")

# Optional: see mismatches
df_mismatch = df_compare[~df_compare['match']]
df_mismatch

PinkWx
Soil_Temp
Total rows compared: 68764
Number of mismatches: 0


,"Date Time, PST",csv_value,db_value,diff,match
